# Pruebas de mejora de `.geo` y `.sif` en Kabre

Este notebook arma una corrida controlada para decidir que cambios dejar fijos en el modelo VAD:

- `annular_rotor`: modelo actual, con imanes internos anulares que abrazan el rotor.
- `central_pill`: variante tipo Ruben/imagen, con imanes internos solidos tipo pastilla dentro del impulsor.
- `AMB visibility`: pares con `I_eval_A = 0` y `I_eval_A = 3` para medir cuanto se ve la bobina sobre cada PMB.
- `SIF checks`: limpieza del solver inactivo, materiales menos ruidosos y una prueba deliberada con Piola apagado.
- `mesh checks`: mismas capas externas fijas, refinando solo la zona interna/rotor y la bobina.

La idea es que corras esto en Kabre, subas los CSV de `reports/`, y con eso fijamos una geometria/malla/SIF antes de seguir con genetico o paralelizacion.


## 0. Configuración

En Kabre dejá `PROJECT = Path('/work/jmorera/Genes')`. Si estás leyendo esto fuera de Kabre, la celda no debe intentar descubrir un repo local: solo va a avisar que la ruta no existe.

Flujo sugerido:

1. `DRY_RUN = True`: revisar rutas y comandos.
2. Escribir templates por variante.
3. Correr preflight Gmsh/ElmerGrid para corregir `ExternalBC`.
4. Escribir casos con `genes/10_poblacion/write_population_cases.py`.
5. Correr barridos pequeños con `genes/20_ejecucion/sweep_mpi.py`.
6. Graficar resultados y decidir qué geometría pasa al genético.

In [ ]:
from pathlib import Path
from datetime import datetime
import importlib.util
import json
import os
import re
import subprocess
import sys

REQUIRED_IMPORTS = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib"}
INSTALL_MISSING = False

missing = [pkg for pkg, module in REQUIRED_IMPORTS.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Faltan paquetes:", missing)
    if INSTALL_MISSING:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    else:
        raise ImportError("Instalá los paquetes faltantes o cambiá INSTALL_MISSING = True.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.25})

In [ ]:
# Configuraci?n principal.
# Si re-ejecut?s esta celda, por defecto conserva RUN_LABEL para no partir el experimento
# entre carpetas distintas. Para crear uno nuevo, pon? FORCE_NEW_EXPERIMENT = True una vez.

DRY_RUN = globals().get("DRY_RUN", True)
FORCE_NEW_EXPERIMENT = False
RESUME_RUN_LABEL = ""  # ejemplo: "geo_sif_trials_20260501_190910" si quer?s retomar uno existente

PROJECT = Path(os.environ.get("KABRE_PROJECT", "/work/jmorera/Genes")).resolve()
LAB_ROOT = PROJECT / "GeneticLab"
GENES_ROOT = PROJECT / "genes"
DOCS_ROOT = PROJECT / "docs"
TEMPLATE_DIR = GENES_ROOT / "00_templates_simulacion"

if RESUME_RUN_LABEL:
    RUN_LABEL = RESUME_RUN_LABEL
elif FORCE_NEW_EXPERIMENT or "RUN_LABEL" not in globals():
    RUN_LABEL = "geo_sif_trials_" + datetime.now().strftime("%Y%m%d_%H%M%S")

EXPERIMENT_ROOT = LAB_ROOT / "experiments" / RUN_LABEL
TEMPLATES_ROOT = EXPERIMENT_ROOT / "templates"
POP_ROOT = EXPERIMENT_ROOT / "results" / "01_population"
CASES_ROOT = EXPERIMENT_ROOT / "cases"
RUNS_ROOT = EXPERIMENT_ROOT / "runs"
REPORTS_ROOT = EXPERIMENT_ROOT / "reports"

WRITE_CASES_PY = GENES_ROOT / "10_poblacion" / "write_population_cases.py"
SWEEP_MPI_PY = GENES_ROOT / "20_ejecucion" / "sweep_mpi.py"
COLLECT_RUNS_PY = GENES_ROOT / "30_postproceso" / "genes_slurm_collect_run_summary.py"

BASE_GEO = TEMPLATE_DIR / "StepsHTX.geo"
BASE_SIF = TEMPLATE_DIR / "P1low.sif"
BASE_DEF = TEMPLATE_DIR / "HMB_circuit.definition"

if PROJECT.exists():
    for folder in [TEMPLATES_ROOT, POP_ROOT, CASES_ROOT, RUNS_ROOT, REPORTS_ROOT]:
        folder.mkdir(parents=True, exist_ok=True)
else:
    print("PROJECT no existe en este kernel:", PROJECT)
    print("Esto est? bien si solo est?s leyendo el notebook fuera de Kabre.")

print("PROJECT:", PROJECT)
print("LAB_ROOT:", LAB_ROOT)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("RUN_LABEL:", RUN_LABEL)
print("DRY_RUN:", DRY_RUN)
for path in [BASE_GEO, BASE_SIF, BASE_DEF, WRITE_CASES_PY, SWEEP_MPI_PY, COLLECT_RUNS_PY]:
    print(path, "->", path.exists())

In [ ]:
MODULE_PREAMBLE = """
module purge
module load elmerfem/9.0
module load gmsh/4.15.0
module load mpich/4.1.1
module load lapack/3.12.1
export LD_LIBRARY_PATH=/work/jmorera/compat_gfortran3:$LD_LIBRARY_PATH
export OMP_NUM_THREADS=1
export OPENBLAS_NUM_THREADS=1
export MKL_NUM_THREADS=1
export NUMEXPR_NUM_THREADS=1
export HYDRA_LAUNCHER=fork
""".strip()

def sh(cmd: str, cwd: Path | None = None, execute: bool = True, use_modules: bool = False) -> str:
    full_cmd = f"{MODULE_PREAMBLE}\n{cmd}" if use_modules else cmd
    if not execute:
        return "DRY_RUN command:\n" + full_cmd
    proc = subprocess.run(["bash", "-lc", full_cmd], cwd=str(cwd) if cwd else None, capture_output=True, text=True)
    chunks = []
    if proc.stdout:
        chunks.append("STDOUT:\n" + proc.stdout)
    if proc.stderr:
        chunks.append("STDERR:\n" + proc.stderr)
    if proc.returncode != 0:
        chunks.append(f"[returncode={proc.returncode}]")
    return "\n".join(chunks)

def should_execute(action_name: str, flag: bool) -> bool:
    execute = bool(flag) and not bool(DRY_RUN)
    print(f"{action_name}: execute={execute} | {action_name}={flag} | DRY_RUN={DRY_RUN}")
    return execute

def experiment_root_for_record(record: dict) -> Path:
    # record['template_dir'] = .../GeneticLab/experiments/<RUN_LABEL>/templates/<scenario>
    # Esto evita mezclar templates de una corrida con cases/manifests de otra si re-ejecut?s config.
    template_dir = Path(record["template_dir"])
    return template_dir.parents[1]

## 1. Referencia de la tesis de Rubén

La celda busca PDFs bajo `docs/` y extrae fragmentos con palabras clave. No decide la geometría automáticamente; sirve para copiar dimensiones o criterios al bloque de parámetros.

In [ ]:
pdf_paths = sorted(DOCS_ROOT.rglob("*.pdf")) if DOCS_ROOT.exists() else []
print("PDFs encontrados:")
for i, path in enumerate(pdf_paths):
    print(f"  [{i}] {path.relative_to(PROJECT)}")

SEARCH_TERMS = ["impulsor", "bobina", "iman", "imán", "permanente", "rotor", "pastilla", "magnet"]

def extract_pdf_snippets(pdf_path: Path, terms=SEARCH_TERMS, max_pages: int = 20) -> pd.DataFrame:
    try:
        from pypdf import PdfReader
    except ImportError:
        try:
            from PyPDF2 import PdfReader
        except ImportError:
            print("No está pypdf/PyPDF2. Instalalo si querés extraer texto del PDF.")
            return pd.DataFrame(columns=["page", "hits", "snippet"])
    reader = PdfReader(str(pdf_path))
    rows = []
    for page_idx, page in enumerate(reader.pages[:max_pages]):
        text = page.extract_text() or ""
        clean = " ".join(text.split())
        lower = clean.lower()
        hits = [term for term in terms if term.lower() in lower]
        if hits:
            rows.append({"page": page_idx + 1, "hits": ", ".join(hits), "snippet": clean[:900]})
    return pd.DataFrame(rows)

thesis_notes = extract_pdf_snippets(pdf_paths[0]) if pdf_paths else pd.DataFrame()
display(thesis_notes)

## 2. Parámetros, familias de prueba y perfiles de malla

Criterio de esta ronda:

- Mantener como referencia el PMB tipo Ruben con pastilla pequena, porque reduce energia pasiva y puede dejar ver mejor el efecto del AMB.
- Probar tambien una pastilla limitada por el radio real del rotor: `r_core = r1 - wall_gap`.
- Incluir una pastilla de area equivalente solo como diagnostico; no necesariamente cabe fisicamente, pero separa el efecto "menos iman" del efecto "cambio de topologia".
- Mantener las cajas externas de malla fijas entre simulaciones. Lo que cambia fino es la region interna/rotor y la bobina.


In [ ]:
BASE_PARAMS_MM = {
    "dx_mm": 0.0, "dy_mm": 0.0, "dz_mm": 0.0,
    "r1_mm": 6.5, "r2_mm": 11.5, "r3_mm": 12.7, "r4_mm": 25.4,
    "h0_mm": 3.2, "h1_mm": 3.4, "gap_z_mm": 1.0,
    "rb1_mm": 12.7, "rb2_mm": 18.7, "hb_mm": 12.7, "gap_pc_mm": 4.0,
    "r_core_mm": 5.2, "rotor_body_r_mm": 11.5,
    "I_eval_A": 3.0, "Ni": 700, "R_Coil1_ohm": 9.0,
}

RUBEN_PILL_R_MM = 5.2
WALL_GAP_MM = 0.5
TIGHT_WALL_GAP_MM = 0.2
PILL_WALL_GAP_R_MM = BASE_PARAMS_MM["r1_mm"] - WALL_GAP_MM
PILL_TIGHT_WALL_R_MM = BASE_PARAMS_MM["r1_mm"] - TIGHT_WALL_GAP_MM
PILL_EQUAL_AREA_R_MM = float(np.sqrt(BASE_PARAMS_MM["r2_mm"] ** 2 - BASE_PARAMS_MM["r1_mm"] ** 2))


def mm(value: float) -> float:
    return float(value) * 1e-3


def with_derived(params: dict) -> dict:
    p = dict(params)
    p["H_pmb_mm"] = 2 * p["h0_mm"] + p["gap_z_mm"]
    p["H_total_mm"] = p["H_pmb_mm"] + p["gap_pc_mm"] + p["hb_mm"]
    p["z_pmb_bottom_mm"] = -p["H_total_mm"] / 2
    p["z_mi1_mm"] = p["z_pmb_bottom_mm"]
    p["z_mi2_mm"] = p["z_mi1_mm"] + p["h0_mm"] + p["gap_z_mm"]
    p["z_me1_mm"] = p["z_mi1_mm"] + (p["h0_mm"] - p["h1_mm"]) / 2
    p["z_me2_mm"] = p["z_me1_mm"] + p["h0_mm"] + p["gap_z_mm"]
    p["z_coil_mm"] = p["z_pmb_bottom_mm"] + p["H_pmb_mm"] + p["gap_pc_mm"]
    p["Ae_Coil1_m2"] = mm(p["rb2_mm"] - p["rb1_mm"]) * mm(p["hb_mm"])
    return p


def inner_area_units(params: dict, variant: str) -> float:
    p = with_derived(params)
    if variant == "annular_rotor":
        return p["r2_mm"] ** 2 - p["r1_mm"] ** 2
    return p["r_core_mm"] ** 2


MESH_PROFILES = {
    "coarse_debug": {
        "air_R_mm": 45.0, "air_H_mm": 52.0,
        "lc_inner_mm": 0.80, "lc_coil_mm": 0.90, "lc_near_mm": 1.40, "lc_mid_mm": 2.20, "lc_far_mm": 4.00,
        "inner_margin_r_mm": 3.0, "inner_margin_z_mm": 2.0, "coil_margin_r_mm": 2.0, "coil_margin_z_mm": 2.0,
        "fixed_near_R_mm": 26.0, "fixed_near_Zmin_mm": -24.0, "fixed_near_Zmax_mm": 18.0,
        "fixed_mid_R_mm": 34.0, "fixed_mid_Zmin_mm": -28.0, "fixed_mid_Zmax_mm": 24.0,
        "algorithm3d": 10, "smoothing": 6, "optimize_threshold": 0.45,
    },
    "balanced_fixed_layers": {
        "air_R_mm": 45.0, "air_H_mm": 52.0,
        "lc_inner_mm": 0.40, "lc_coil_mm": 0.55, "lc_near_mm": 0.90, "lc_mid_mm": 1.80, "lc_far_mm": 3.20,
        "inner_margin_r_mm": 2.5, "inner_margin_z_mm": 1.5, "coil_margin_r_mm": 2.0, "coil_margin_z_mm": 2.0,
        "fixed_near_R_mm": 26.0, "fixed_near_Zmin_mm": -24.0, "fixed_near_Zmax_mm": 18.0,
        "fixed_mid_R_mm": 36.0, "fixed_mid_Zmin_mm": -30.0, "fixed_mid_Zmax_mm": 26.0,
        "algorithm3d": 10, "smoothing": 8, "optimize_threshold": 0.45,
    },
    "focus_inner_coil": {
        "air_R_mm": 45.0, "air_H_mm": 52.0,
        "lc_inner_mm": 0.30, "lc_coil_mm": 0.42, "lc_near_mm": 0.90, "lc_mid_mm": 1.80, "lc_far_mm": 3.20,
        "inner_margin_r_mm": 2.5, "inner_margin_z_mm": 1.5, "coil_margin_r_mm": 2.0, "coil_margin_z_mm": 2.0,
        "fixed_near_R_mm": 26.0, "fixed_near_Zmin_mm": -24.0, "fixed_near_Zmax_mm": 18.0,
        "fixed_mid_R_mm": 36.0, "fixed_mid_Zmin_mm": -30.0, "fixed_mid_Zmax_mm": 26.0,
        "algorithm3d": 10, "smoothing": 8, "optimize_threshold": 0.45,
    },
    "balanced_delaunay": {
        "air_R_mm": 45.0, "air_H_mm": 52.0,
        "lc_inner_mm": 0.40, "lc_coil_mm": 0.55, "lc_near_mm": 0.90, "lc_mid_mm": 1.80, "lc_far_mm": 3.20,
        "inner_margin_r_mm": 2.5, "inner_margin_z_mm": 1.5, "coil_margin_r_mm": 2.0, "coil_margin_z_mm": 2.0,
        "fixed_near_R_mm": 26.0, "fixed_near_Zmin_mm": -24.0, "fixed_near_Zmax_mm": 18.0,
        "fixed_mid_R_mm": 36.0, "fixed_mid_Zmin_mm": -30.0, "fixed_mid_Zmax_mm": 26.0,
        "algorithm3d": 1, "smoothing": 8, "optimize_threshold": 0.45,
    },
    "fine_inner_coil": {
        "air_R_mm": 50.0, "air_H_mm": 60.0,
        "lc_inner_mm": 0.28, "lc_coil_mm": 0.40, "lc_near_mm": 0.75, "lc_mid_mm": 1.50, "lc_far_mm": 3.00,
        "inner_margin_r_mm": 2.5, "inner_margin_z_mm": 1.5, "coil_margin_r_mm": 2.0, "coil_margin_z_mm": 2.0,
        "fixed_near_R_mm": 28.0, "fixed_near_Zmin_mm": -26.0, "fixed_near_Zmax_mm": 20.0,
        "fixed_mid_R_mm": 40.0, "fixed_mid_Zmin_mm": -34.0, "fixed_mid_Zmax_mm": 30.0,
        "algorithm3d": 10, "smoothing": 8, "optimize_threshold": 0.45,
    },
}

display(pd.DataFrame(MESH_PROFILES).T)
print(f"Pastilla Ruben: r={RUBEN_PILL_R_MM:.2f} mm")
print(f"Pastilla con pared {WALL_GAP_MM:.1f} mm: r={PILL_WALL_GAP_R_MM:.2f} mm")
print(f"Pastilla area-equivalente al anillo: r={PILL_EQUAL_AREA_R_MM:.2f} mm")


In [ ]:
def add_rect(ax, r0, r1, z0, h, label, color, alpha=0.75, hatch=None):
    ax.add_patch(Rectangle((r0, z0), r1 - r0, h, facecolor=color, edgecolor="black", alpha=alpha, hatch=hatch, label=label))


def plot_rz_geometry(params: dict, variant: str, mesh_profile: str = "balanced_fixed_layers", ax=None):
    p = with_derived(params)
    mesh = MESH_PROFILES[mesh_profile]
    colors = {"inner": "#4C78A8", "outer": "#F58518", "coil": "#54A24B", "rotor": "#999999", "layer": "#B279A2"}
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 7))
    add_rect(ax, 0, mesh["fixed_mid_R_mm"], mesh["fixed_mid_Zmin_mm"], mesh["fixed_mid_Zmax_mm"] - mesh["fixed_mid_Zmin_mm"], "fixed mid", colors["layer"], 0.08)
    add_rect(ax, 0, mesh["fixed_near_R_mm"], mesh["fixed_near_Zmin_mm"], mesh["fixed_near_Zmax_mm"] - mesh["fixed_near_Zmin_mm"], "fixed near", colors["layer"], 0.12)
    add_rect(ax, 0, p["rotor_body_r_mm"], p["z_pmb_bottom_mm"], p["H_pmb_mm"], "rotor envelope", colors["rotor"], 0.12, hatch="//")
    add_rect(ax, p["r3_mm"], p["r4_mm"], p["z_me1_mm"], p["h1_mm"], "outer magnets", colors["outer"], 0.65)
    add_rect(ax, p["r3_mm"], p["r4_mm"], p["z_me2_mm"], p["h1_mm"], "outer magnets", colors["outer"], 0.65)
    if variant == "annular_rotor":
        add_rect(ax, p["r1_mm"], p["r2_mm"], p["z_mi1_mm"], p["h0_mm"], "inner annular", colors["inner"], 0.75)
        add_rect(ax, p["r1_mm"], p["r2_mm"], p["z_mi2_mm"], p["h0_mm"], "inner annular", colors["inner"], 0.75)
    else:
        add_rect(ax, 0, p["r_core_mm"], p["z_mi1_mm"], p["h0_mm"], "inner pill", colors["inner"], 0.75)
        add_rect(ax, 0, p["r_core_mm"], p["z_mi2_mm"], p["h0_mm"], "inner pill", colors["inner"], 0.75)
    add_rect(ax, p["rb1_mm"], p["rb2_mm"], p["z_coil_mm"], p["hb_mm"], "coil", colors["coil"], 0.55, hatch="xx")
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("radio r [mm]")
    ax.set_ylabel("z [mm]")
    ax.set_title(f"{variant}\nr_core={p['r_core_mm']:.2f} mm | {mesh_profile}")
    ax.set_xlim(0, max(mesh["air_R_mm"], p["r4_mm"]) * 1.05)
    ax.set_ylim(-mesh["air_H_mm"] / 2 * 1.05, mesh["air_H_mm"] / 2 * 1.05)
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(unique.values(), unique.keys(), loc="upper right", fontsize=8)
    return ax


geom_refs = [
    {"label": "Anillo actual", "variant": "annular_rotor", "r_core_mm": np.nan, "params": dict(BASE_PARAMS_MM)},
    {"label": "Pastilla Ruben", "variant": "central_pill", "r_core_mm": RUBEN_PILL_R_MM, "params": {**BASE_PARAMS_MM, "r_core_mm": RUBEN_PILL_R_MM}},
    {"label": "Pastilla r1-wall", "variant": "central_pill", "r_core_mm": PILL_WALL_GAP_R_MM, "params": {**BASE_PARAMS_MM, "r_core_mm": PILL_WALL_GAP_R_MM}},
    {"label": "Pastilla area eq.", "variant": "central_pill", "r_core_mm": PILL_EQUAL_AREA_R_MM, "params": {**BASE_PARAMS_MM, "r_core_mm": PILL_EQUAL_AREA_R_MM}},
]
ann_area = inner_area_units(BASE_PARAMS_MM, "annular_rotor")
geom_table = pd.DataFrame([
    {
        "geometry": g["label"],
        "variant": g["variant"],
        "r_core_mm": g["r_core_mm"],
        "inner_area_units": inner_area_units(g["params"], g["variant"]),
        "area_vs_annular": inner_area_units(g["params"], g["variant"]) / ann_area,
    }
    for g in geom_refs
])
display(geom_table)

fig, axes = plt.subplots(1, len(geom_refs), figsize=(18, 5), sharey=True)
for ax, g in zip(axes, geom_refs):
    plot_rz_geometry(g["params"], g["variant"], ax=ax)
    ax.set_title(g["label"])
plt.tight_layout()


## 3. Generar templates por variante

El `.geo` generado conserva `Physical Volume` para los cuerpos `1,3,5,7,20,30`. El `ExternalBC` se corrige luego con preflight, porque las fronteras sí pueden cambiar.

In [ ]:
def fmt_m(value_mm: float) -> str:
    return f"{mm(value_mm):.12g}"


def render_geo(params: dict, variant: str, mesh: dict) -> str:
    p = with_derived(params)
    if variant == "annular_rotor":
        inner = """
Cylinder(5) = {dx, dy, z_mi1+dz,  0,0,h0,  r2, 2*Pi};
Cylinder(6) = {dx, dy, z_mi1+dz,  0,0,h0,  r1, 2*Pi};
MagIntInf[] = BooleanDifference{ Volume{5}; Delete; }{ Volume{6}; Delete; };
InnerMag1 = MagIntInf[0];

Cylinder(7) = {dx, dy, z_mi2+dz,  0,0,h0,  r2, 2*Pi};
Cylinder(8) = {dx, dy, z_mi2+dz,  0,0,h0,  r1, 2*Pi};
MagIntSup[] = BooleanDifference{ Volume{7}; Delete; }{ Volume{8}; Delete; };
InnerMag2 = MagIntSup[0];
""".strip()
        magnetic_outer_r = p["r2_mm"]
    else:
        inner = """
Cylinder(5) = {dx, dy, z_mi1+dz,  0,0,h0,  r_core, 2*Pi};
InnerMag1 = 5;

Cylinder(7) = {dx, dy, z_mi2+dz,  0,0,h0,  r_core, 2*Pi};
InnerMag2 = 7;
""".strip()
        magnetic_outer_r = p["r_core_mm"]

    # La caja fina cubre al menos el envolvente fisico del rotor, aunque la pastilla sea menor.
    # Asi la comparacion no gana elementos solo por hacer mas chico el iman interno.
    inner_ref_r = max(magnetic_outer_r, p["rotor_body_r_mm"]) + mesh["inner_margin_r_mm"]
    z_inner_min = p["z_mi1_mm"] - mesh["inner_margin_z_mm"]
    z_inner_max = p["z_mi2_mm"] + p["h0_mm"] + mesh["inner_margin_z_mm"]
    coil_ref_r = p["rb2_mm"] + mesh["coil_margin_r_mm"]
    z_coil_min = p["z_coil_mm"] - mesh["coil_margin_z_mm"]
    z_coil_max = p["z_coil_mm"] + p["hb_mm"] + mesh["coil_margin_z_mm"]
    lc_min = min(mesh["lc_inner_mm"], mesh["lc_coil_mm"], mesh["lc_near_mm"], mesh["lc_mid_mm"], mesh["lc_far_mm"])

    return f"""
SetFactory("OpenCASCADE");

use_coil = 1;
dx = {fmt_m(p['dx_mm'])};
dy = {fmt_m(p['dy_mm'])};
dz = {fmt_m(p['dz_mm'])};
r1 = {fmt_m(p['r1_mm'])};
r2 = {fmt_m(p['r2_mm'])};
r3 = {fmt_m(p['r3_mm'])};
r4 = {fmt_m(p['r4_mm'])};
r_core = {fmt_m(p['r_core_mm'])};
h0 = {fmt_m(p['h0_mm'])};
h1 = {fmt_m(p['h1_mm'])};
gap_z = {fmt_m(p['gap_z_mm'])};
rb1 = {fmt_m(p['rb1_mm'])};
rb2 = {fmt_m(p['rb2_mm'])};
hb  = {fmt_m(p['hb_mm'])};
gap_pc = {fmt_m(p['gap_pc_mm'])};

H_pmb = 2*h0 + gap_z;
H_total = H_pmb + gap_pc + hb;
z_pmb_bottom = -H_total/2;
air_R = {fmt_m(mesh['air_R_mm'])};
air_H = {fmt_m(mesh['air_H_mm'])};
z_air = -air_H/2;

z_mi1 = z_pmb_bottom;
z_mi2 = z_mi1 + h0 + gap_z;
{inner}

z_me1 = z_mi1 + (h0 - h1)/2;
Cylinder(1) = {{0,0,z_me1, 0,0,h1, r4, 2*Pi}};
Cylinder(2) = {{0,0,z_me1, 0,0,h1, r3, 2*Pi}};
MagExtInf[] = BooleanDifference{{ Volume{{1}}; Delete; }}{{ Volume{{2}}; Delete; }};
OuterMag1 = MagExtInf[0];
z_me2 = z_me1 + h0 + gap_z;
Cylinder(3) = {{0,0,z_me2, 0,0,h1, r4, 2*Pi}};
Cylinder(4) = {{0,0,z_me2, 0,0,h1, r3, 2*Pi}};
MagExtSup[] = BooleanDifference{{ Volume{{3}}; Delete; }}{{ Volume{{4}}; Delete; }};
OuterMag2 = MagExtSup[0];

z_coil = z_pmb_bottom + H_pmb + gap_pc;
Cylinder(20) = {{0,0,z_coil, 0,0,hb, rb2, 2*Pi}};
Cylinder(21) = {{0,0,z_coil, 0,0,hb, rb1, 2*Pi}};
CoilVol[] = BooleanDifference{{ Volume{{20}}; Delete; }}{{ Volume{{21}}; Delete; }};
Coil = CoilVol[0];

Cylinder(30) = {{0,0,z_air, 0,0,air_H, air_R, 2*Pi}};
AirClean[] = BooleanDifference{{ Volume{{30}}; Delete; }}{{ Volume{{OuterMag1, OuterMag2, InnerMag1, InnerMag2, Coil}}; }};
Air = AirClean[0];
Coherence;

Physical Volume("OuterMag1", 1) = {{OuterMag1}};
Physical Volume("OuterMag2", 3) = {{OuterMag2}};
Physical Volume("InnerMag1", 5) = {{InnerMag1}};
Physical Volume("InnerMag2", 7) = {{InnerMag2}};
Physical Volume("Coil", 20) = {{Coil}};
Physical Volume("Air", 30) = {{Air}};

Mesh.Algorithm3D = {int(mesh.get('algorithm3d', 10))};
Mesh.Smoothing = {int(mesh.get('smoothing', 8))};
Mesh.Optimize = 1;
Mesh.OptimizeNetgen = 1;
Mesh.OptimizeThreshold = {float(mesh.get('optimize_threshold', 0.45)):.6g};
lc_inner = {fmt_m(mesh['lc_inner_mm'])};
lc_coil = {fmt_m(mesh['lc_coil_mm'])};
lc_near = {fmt_m(mesh['lc_near_mm'])};
lc_mid = {fmt_m(mesh['lc_mid_mm'])};
lc_far = {fmt_m(mesh['lc_far_mm'])};
Mesh.CharacteristicLengthMin = {fmt_m(lc_min)};
Mesh.CharacteristicLengthMax = lc_far;

Field[10] = Box;
Field[10].VIn = lc_inner; Field[10].VOut = lc_far;
Field[10].XMin = -{fmt_m(inner_ref_r)}; Field[10].XMax = {fmt_m(inner_ref_r)};
Field[10].YMin = -{fmt_m(inner_ref_r)}; Field[10].YMax = {fmt_m(inner_ref_r)};
Field[10].ZMin = {fmt_m(z_inner_min)}; Field[10].ZMax = {fmt_m(z_inner_max)};

Field[11] = Box;
Field[11].VIn = lc_coil; Field[11].VOut = lc_far;
Field[11].XMin = -{fmt_m(coil_ref_r)}; Field[11].XMax = {fmt_m(coil_ref_r)};
Field[11].YMin = -{fmt_m(coil_ref_r)}; Field[11].YMax = {fmt_m(coil_ref_r)};
Field[11].ZMin = {fmt_m(z_coil_min)}; Field[11].ZMax = {fmt_m(z_coil_max)};

Field[12] = Box;
Field[12].VIn = lc_near; Field[12].VOut = lc_far;
Field[12].XMin = -{fmt_m(mesh['fixed_near_R_mm'])}; Field[12].XMax = {fmt_m(mesh['fixed_near_R_mm'])};
Field[12].YMin = -{fmt_m(mesh['fixed_near_R_mm'])}; Field[12].YMax = {fmt_m(mesh['fixed_near_R_mm'])};
Field[12].ZMin = {fmt_m(mesh['fixed_near_Zmin_mm'])}; Field[12].ZMax = {fmt_m(mesh['fixed_near_Zmax_mm'])};

Field[13] = Box;
Field[13].VIn = lc_mid; Field[13].VOut = lc_far;
Field[13].XMin = -{fmt_m(mesh['fixed_mid_R_mm'])}; Field[13].XMax = {fmt_m(mesh['fixed_mid_R_mm'])};
Field[13].YMin = -{fmt_m(mesh['fixed_mid_R_mm'])}; Field[13].YMax = {fmt_m(mesh['fixed_mid_R_mm'])};
Field[13].ZMin = {fmt_m(mesh['fixed_mid_Zmin_mm'])}; Field[13].ZMax = {fmt_m(mesh['fixed_mid_Zmax_mm'])};

Field[14] = Min;
Field[14].FieldsList = {{10,11,12,13}};
Background Field = 14;
Mesh.MshFileVersion = 2.2;
Mesh.Binary = 0;
Mesh.SaveParametric = 0;
Mesh.SaveAll = 1;
Mesh 3;
""".strip() + "\n"


In [ ]:
def replace_definition_assignment(text: str, var: str, value: str) -> str:
    pat = re.compile(rf"(?m)^(\s*\$?\s*{re.escape(var)}\s*=\s*)([^\n\r!#;]*)(.*)$")
    if not pat.search(text):
        raise RuntimeError(f"No encontre {var} en .definition")
    return pat.sub(rf"\g<1>{value}\g<3>", text, count=1)


def render_definition(params: dict, base_text: str) -> str:
    p = with_derived(params)
    text = replace_definition_assignment(base_text, "I1", f"{p['I_eval_A']:.12g}")
    text = replace_definition_assignment(text, "N_Coil1", f"{int(p['Ni'])}")
    text = replace_definition_assignment(text, "Ae_Coil1", f"{p['Ae_Coil1_m2']:.12g}")
    return text


def update_external_bc(sif_text: str, boundary_ids: list[int]) -> str:
    ids = " ".join(str(int(x)) for x in boundary_ids)
    updated = re.sub(r"Target Boundaries\(\d+\)\s*=\s*[0-9 ]+", f"Target Boundaries({len(boundary_ids)}) = {ids}", sif_text)
    if updated == sif_text:
        raise ValueError("No encontre Target Boundaries(...) en el SIF")
    return updated


def clean_inactive_solver5(sif_text: str) -> str:
    text = re.sub(r"Active Solvers\(5\)\s*=\s*1\s+2\s+3\s+4\s+5", "Active Solvers(4) = 1 2 3 4", sif_text)
    text = re.sub(r"Active Solvers\(3\)\s*=\s*3\s+4\s+5", "Active Solvers(2) = 3 4", text)
    return text


def set_piola_transform(sif_text: str, enabled: bool) -> str:
    value = "True" if enabled else "False"
    text, n = re.subn(r"Use Piola Transform\s*=\s*Logical\s+(True|False)", f"Use Piola Transform = Logical {value}", sif_text)
    if n == 0:
        raise ValueError("No encontre 'Use Piola Transform' en el SIF")
    return text


def ensure_material_permittivity(sif_text: str, material_ids=(2, 3, 4), value="1.0") -> str:
    def patch_block(match):
        mat_id = int(match.group(1))
        block = match.group(0)
        if mat_id not in material_ids or re.search(r"Relative Permittivity\s*=", block):
            return block
        return re.sub(r"\nEnd\s*$", f"\n  Relative Permittivity = {value}\nEnd", block)

    return re.sub(r"(?ms)^Material\s+(\d+)\s*\n.*?^End\s*$", patch_block, sif_text)


def render_sif(base_text: str, sif_mode: str = "base") -> str:
    text = base_text
    if "clean_solver5" in sif_mode:
        text = clean_inactive_solver5(text)
    if "quiet_materials" in sif_mode:
        text = ensure_material_permittivity(text)
    if "piola_off" in sif_mode:
        text = set_piola_transform(text, False)
    elif "piola_on" in sif_mode:
        text = set_piola_transform(text, True)
    return text


## 4. Plan de experimentos

La tabla se arma desde un catalogo y `ENABLED_GROUPS`. Para una primera corrida completa yo dejaria todo activo excepto `diagnostic_equal_area`; si Kabre esta muy cargado, corre primero `geometry_screen` + `amb_visibility` y luego seguimos con malla/SIF.


In [ ]:
ENABLED_GROUPS = {
    "geometry_screen": True,
    "amb_visibility": True,
    "mesh_check": True,
    "sif_check": True,
    "diagnostic_equal_area": False,
}

DEFAULT_SIF_MODE = "clean_solver5_quiet_materials_piola_on"
BASELINE_MESH = "balanced_fixed_layers"
BASELINE_I_A = 3.0
ZERO_I_A = 0.0

SCENARIO_CATALOG = [
    {"group": "geometry_screen", "name": "A_annular_real_I3_balanced", "geometry_key": "annular_real", "variant": "annular_rotor", "mesh_profile": BASELINE_MESH, "r_core_mm": np.nan, "I_eval_A": BASELINE_I_A, "sif_mode": DEFAULT_SIF_MODE, "reason": "anillo actual con radio real"},
    {"group": "geometry_screen", "name": "B_pill_ruben_I3_balanced", "geometry_key": "pill_ruben", "variant": "central_pill", "mesh_profile": BASELINE_MESH, "r_core_mm": RUBEN_PILL_R_MM, "I_eval_A": BASELINE_I_A, "sif_mode": DEFAULT_SIF_MODE, "reason": "PMB tipo Ruben/imagen"},
    {"group": "geometry_screen", "name": "B_pill_wallgap0p5_I3_balanced", "geometry_key": "pill_wallgap0p5", "variant": "central_pill", "mesh_profile": BASELINE_MESH, "r_core_mm": PILL_WALL_GAP_R_MM, "I_eval_A": BASELINE_I_A, "sif_mode": DEFAULT_SIF_MODE, "reason": "r_core = r1 - 0.5 mm"},
    {"group": "geometry_screen", "name": "B_pill_wallgap0p2_I3_balanced", "geometry_key": "pill_wallgap0p2", "variant": "central_pill", "mesh_profile": BASELINE_MESH, "r_core_mm": PILL_TIGHT_WALL_R_MM, "I_eval_A": BASELINE_I_A, "sif_mode": DEFAULT_SIF_MODE, "reason": "r_core = r1 - 0.2 mm"},

    {"group": "amb_visibility", "name": "A_annular_real_I0_balanced", "geometry_key": "annular_real", "variant": "annular_rotor", "mesh_profile": BASELINE_MESH, "r_core_mm": np.nan, "I_eval_A": ZERO_I_A, "sif_mode": DEFAULT_SIF_MODE, "reason": "PMB-only aproximado para delta AMB"},
    {"group": "amb_visibility", "name": "B_pill_ruben_I0_balanced", "geometry_key": "pill_ruben", "variant": "central_pill", "mesh_profile": BASELINE_MESH, "r_core_mm": RUBEN_PILL_R_MM, "I_eval_A": ZERO_I_A, "sif_mode": DEFAULT_SIF_MODE, "reason": "PMB-only aproximado para delta AMB"},
    {"group": "amb_visibility", "name": "B_pill_wallgap0p5_I0_balanced", "geometry_key": "pill_wallgap0p5", "variant": "central_pill", "mesh_profile": BASELINE_MESH, "r_core_mm": PILL_WALL_GAP_R_MM, "I_eval_A": ZERO_I_A, "sif_mode": DEFAULT_SIF_MODE, "reason": "PMB-only aproximado para delta AMB"},

    {"group": "mesh_check", "name": "A_annular_real_I3_focus", "geometry_key": "annular_real", "variant": "annular_rotor", "mesh_profile": "focus_inner_coil", "r_core_mm": np.nan, "I_eval_A": BASELINE_I_A, "sif_mode": DEFAULT_SIF_MODE, "reason": "malla focal interna/bobina"},
    {"group": "mesh_check", "name": "B_pill_wallgap0p5_I3_focus", "geometry_key": "pill_wallgap0p5", "variant": "central_pill", "mesh_profile": "focus_inner_coil", "r_core_mm": PILL_WALL_GAP_R_MM, "I_eval_A": BASELINE_I_A, "sif_mode": DEFAULT_SIF_MODE, "reason": "malla focal interna/bobina"},
    {"group": "mesh_check", "name": "B_pill_ruben_I3_focus", "geometry_key": "pill_ruben", "variant": "central_pill", "mesh_profile": "focus_inner_coil", "r_core_mm": RUBEN_PILL_R_MM, "I_eval_A": BASELINE_I_A, "sif_mode": DEFAULT_SIF_MODE, "reason": "malla focal interna/bobina"},

    {"group": "sif_check", "name": "S_annular_baseSIF_I3_balanced", "geometry_key": "annular_real", "variant": "annular_rotor", "mesh_profile": BASELINE_MESH, "r_core_mm": np.nan, "I_eval_A": BASELINE_I_A, "sif_mode": "base", "reason": "comparar contra SIF original"},
    {"group": "sif_check", "name": "S_annular_piolaOff_I3_balanced", "geometry_key": "annular_real", "variant": "annular_rotor", "mesh_profile": BASELINE_MESH, "r_core_mm": np.nan, "I_eval_A": BASELINE_I_A, "sif_mode": "clean_solver5_quiet_materials_piola_off", "reason": "sensibilidad Piola; no candidato por defecto"},
    {"group": "sif_check", "name": "S_pill_wallgap0p5_baseSIF_I3_balanced", "geometry_key": "pill_wallgap0p5", "variant": "central_pill", "mesh_profile": BASELINE_MESH, "r_core_mm": PILL_WALL_GAP_R_MM, "I_eval_A": BASELINE_I_A, "sif_mode": "base", "reason": "efecto de limpiar SIF en pastilla"},

    {"group": "diagnostic_equal_area", "name": "D_pill_equalarea_I3_balanced", "geometry_key": "pill_equalarea", "variant": "central_pill", "mesh_profile": BASELINE_MESH, "r_core_mm": PILL_EQUAL_AREA_R_MM, "I_eval_A": BASELINE_I_A, "sif_mode": DEFAULT_SIF_MODE, "reason": "misma area magnetica que anillo; diagnostico"},
]

SCENARIOS = pd.DataFrame([row for row in SCENARIO_CATALOG if ENABLED_GROUPS.get(row["group"], False)])
SCENARIOS["inner_area_units"] = SCENARIOS.apply(
    lambda r: inner_area_units(
        {**BASE_PARAMS_MM, "r_core_mm": BASE_PARAMS_MM["r_core_mm"] if pd.isna(r["r_core_mm"]) else r["r_core_mm"]},
        r["variant"],
    ),
    axis=1,
)
SCENARIOS["area_vs_annular"] = SCENARIOS["inner_area_units"] / inner_area_units(BASE_PARAMS_MM, "annular_rotor")
SCENARIOS = SCENARIOS.sort_values(["group", "geometry_key", "I_eval_A", "mesh_profile", "sif_mode", "name"]).reset_index(drop=True)
display(SCENARIOS)

fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(SCENARIOS))))
plot_df = SCENARIOS.sort_values("area_vs_annular")
ax.barh(plot_df["name"], plot_df["area_vs_annular"], color=np.where(plot_df["variant"].eq("annular_rotor"), "#4C78A8", "#F58518"))
ax.axvline(1.0, color="black", linewidth=1)
ax.set_xlabel("area magnetica interna relativa al anillo actual")
ax.set_ylabel("")
plt.tight_layout()


In [ ]:
def scenario_params(row: pd.Series) -> dict:
    p = dict(BASE_PARAMS_MM)
    overrides = [
        "r_core_mm", "I_eval_A", "Ni", "R_Coil1_ohm",
        "rb1_mm", "rb2_mm", "hb_mm", "gap_pc_mm",
        "r1_mm", "r2_mm", "r3_mm", "r4_mm", "h0_mm", "h1_mm", "gap_z_mm",
    ]
    for key in overrides:
        if key in row and not pd.isna(row[key]):
            p[key] = float(row[key])
    return with_derived(p)


def write_scenario_template(row: pd.Series) -> dict:
    if not BASE_SIF.exists() or not BASE_DEF.exists():
        raise FileNotFoundError("Faltan templates base en genes/00_templates_simulacion")
    params = scenario_params(row)
    mesh = MESH_PROFILES[row["mesh_profile"]]
    scenario_dir = TEMPLATES_ROOT / row["name"]
    scenario_dir.mkdir(parents=True, exist_ok=True)
    geo_path = scenario_dir / "StepsHTX.geo"
    sif_path = scenario_dir / "P1low.sif"
    def_path = scenario_dir / "HMB_circuit.definition"
    base_sif_text = BASE_SIF.read_text(encoding="utf-8", errors="ignore")
    geo_path.write_text(render_geo(params, row["variant"], mesh), encoding="utf-8")
    sif_path.write_text(render_sif(base_sif_text, row.get("sif_mode", "base")), encoding="utf-8")
    def_path.write_text(render_definition(params, BASE_DEF.read_text(encoding="utf-8", errors="ignore")), encoding="utf-8")
    config = {"scenario": row.to_dict(), "params_mm": params, "mesh": mesh}
    (scenario_dir / "scenario_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    return {"scenario": row["name"], "template_dir": scenario_dir, "geo": geo_path, "sif": sif_path, "definition": def_path}


def write_one_row_population(row: pd.Series) -> Path:
    params = scenario_params(row)
    pop = pd.DataFrame([{
        "individual_id": row["name"], "generation": 0, "algorithm": "geo_sif_trial",
        "trial_group": row.get("group", ""), "geometry_key": row.get("geometry_key", ""),
        "rb1_mm": params["rb1_mm"], "rb2_mm": params["rb2_mm"], "hb_mm": params["hb_mm"], "gap_pc_mm": params["gap_pc_mm"],
        "I_eval_A": params["I_eval_A"], "Ni": params["Ni"],
        "geometry_variant": row["variant"], "mesh_profile": row["mesh_profile"], "sif_mode": row.get("sif_mode", "base"),
        "r_core_mm": params["r_core_mm"], "area_vs_annular": row.get("area_vs_annular", np.nan),
    }])
    out = POP_ROOT / f"{row['name']}_population.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    pop.to_csv(out, index=False)
    return out


WRITE_TEMPLATES = True
scenario_records = []
if WRITE_TEMPLATES:
    for _, row in SCENARIOS.iterrows():
        record = write_scenario_template(row)
        record["population_csv"] = write_one_row_population(row)
        scenario_records.append(record)
    display(pd.DataFrame([{k: str(v) for k, v in rec.items()} for rec in scenario_records]))


## 5. Preflight y `ExternalBC`

Este paso corre Gmsh + ElmerGrid para cada template, busca superficies 2D externas del aire (`30-exterior`) y actualiza el `P1low.sif` del template. Dejalo en `RUN_PREFLIGHT=False` hasta estar en Kabre y haber revisado los comandos.

In [ ]:
def load_elmer_mesh_maps(mesh_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    elem_body, body_counts = {}, {}
    with (mesh_dir / "mesh.elements").open(encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.split()
            if len(parts) < 3:
                continue
            eid, body = int(parts[0]), int(parts[1])
            elem_body[eid] = body
            body_counts[body] = body_counts.get(body, 0) + 1
    rows = []
    with (mesh_dir / "mesh.boundary").open(encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.split()
            if len(parts) < 6:
                continue
            parent_a = elem_body.get(int(parts[2]), 0)
            parent_b = elem_body.get(int(parts[3]), 0) if int(parts[3]) > 0 else 0
            pair = "-".join(map(str, sorted([parent_a, parent_b]))) if parent_b > 0 else f"{parent_a}-exterior"
            rows.append({"Boundary": int(parts[1]), "ElementType": int(parts[4]), "ParentPairs": pair})
    bodies = pd.DataFrame([{"Body": k, "Elements": v} for k, v in sorted(body_counts.items())])
    boundaries = pd.DataFrame(rows)
    if not boundaries.empty:
        boundaries = boundaries.groupby(["Boundary", "ElementType", "ParentPairs"], as_index=False).size().rename(columns={"size": "Elements"})
    return bodies, boundaries


def external_air_boundary_ids(boundaries: pd.DataFrame, air_body: int = 30) -> list[int]:
    if boundaries.empty:
        return []
    mask = (boundaries["ParentPairs"] == f"{air_body}-exterior") & (boundaries["ElementType"] == 303)
    return sorted(boundaries.loc[mask, "Boundary"].astype(int).unique().tolist())


def preflight_template(template_dir: Path):
    out = template_dir / "preflight"
    cmd = f"""
set -euo pipefail
rm -rf {out}
mkdir -p {out}
gmsh {template_dir / 'StepsHTX.geo'} -3 -format msh2 -save_all -o {out / 'mesh.msh'}
cd {out}
ElmerGrid 14 2 mesh.msh -out mesh
"""
    print(sh(cmd, cwd=PROJECT, execute=should_execute("RUN_PREFLIGHT", RUN_PREFLIGHT), use_modules=True))
    if DRY_RUN or not RUN_PREFLIGHT:
        return None, None, []
    bodies, boundaries = load_elmer_mesh_maps(out / "mesh")
    ids = external_air_boundary_ids(boundaries)
    if not ids:
        raise RuntimeError(f"No encontre superficies externas de Air en {template_dir}")
    sif_path = template_dir / "P1low.sif"
    sif_path.write_text(update_external_bc(sif_path.read_text(encoding="utf-8", errors="ignore"), ids), encoding="utf-8")
    bodies.to_csv(out / "body_counts.csv", index=False)
    boundaries.to_csv(out / "boundary_report.csv", index=False)
    external_elements = int(boundaries.loc[boundaries["Boundary"].isin(ids), "Elements"].sum())
    return bodies, boundaries, ids, external_elements


RUN_PREFLIGHT = not DRY_RUN
if RUN_PREFLIGHT:
    summary = []
    for rec in scenario_records:
        bodies, boundaries, ids, external_elements = preflight_template(Path(rec["template_dir"]))
        summary.append({"scenario": rec["scenario"], "external_bc": ids, "external_surface_elements": external_elements, "n_bodies": len(bodies), "volume_elements": int(bodies["Elements"].sum())})
    preflight_summary = pd.DataFrame(summary)
    preflight_summary.to_csv(REPORTS_ROOT / "preflight_summary.csv", index=False)
    display(preflight_summary)
else:
    print("RUN_PREFLIGHT=False. Cambialo a True en Kabre para corregir ExternalBC antes de escribir/correr casos.")


## 6. Escribir casos con `write_population_cases.py`

In [ ]:
def write_cases_command(record: dict) -> tuple[str, Path]:
    scenario = record["scenario"]
    exp_root = experiment_root_for_record(record)
    manifest = exp_root / "results" / "01_population" / f"{scenario}_manifest.csv"
    outdir = exp_root / "cases" / scenario
    cmd = f"""
python3 {WRITE_CASES_PY} \
  --population {record['population_csv']} \
  --geo-template {record['geo']} \
  --sif-template {record['sif']} \
  --definition-template {record['definition']} \
  --outdir {outdir} \
  --manifest-out {manifest}
"""
    return cmd, manifest

RUN_WRITE_CASES = not DRY_RUN
manifest_paths = []
execute_write_cases = should_execute("RUN_WRITE_CASES", RUN_WRITE_CASES)

for rec in scenario_records:
    cmd, manifest = write_cases_command(rec)
    print("\n###", rec["scenario"])
    print(sh(cmd, cwd=PROJECT, execute=execute_write_cases, use_modules=False))
    manifest_paths.append(manifest)

if execute_write_cases:
    for manifest in manifest_paths:
        print(manifest, "->", manifest.exists())

## 7. Barrido corto por escenario

Uso recomendado para esta ronda: `-1.0, -0.5, 0, 0.5, 1.0 mm`. Con cinco puntos ya podemos ajustar una parabola y no depender tanto de una derivada central de solo tres puntos.


In [ ]:
START_MM = -1.0
END_MM = 1.0
STEP_MM = 0.5
INNER_NPROCS = 20
LAUNCHER = "mpirun"
PARTITION_METHOD = "metiskway"
BIND = "core"
RUN_SWEEPS = not DRY_RUN
execute_sweeps = should_execute("RUN_SWEEPS", RUN_SWEEPS)

for rec in scenario_records:
    scenario = rec["scenario"]
    exp_root = experiment_root_for_record(rec)
    case_dir = exp_root / "cases" / scenario / scenario
    outdir = exp_root / "runs" / scenario
    cmd = f"""
python3 {SWEEP_MPI_PY} \
  --geo {case_dir / 'StepsHTX.geo'} \
  --sif {case_dir / 'P1low.sif'} \
  --definition {case_dir / 'HMB_circuit.definition'} \
  --outdir {outdir} \
  --axis dz \
  --start-mm {START_MM} \
  --end-mm {END_MM} \
  --step-mm {STEP_MM} \
  --nprocs {INNER_NPROCS} \
  --partition-method {PARTITION_METHOD} \
  --launcher {LAUNCHER} \
  --bind {BIND}
"""
    print("\n###", scenario)
    print(sh(cmd, cwd=PROJECT, execute=execute_sweeps, use_modules=True))


## 8. Graficar resultados existentes

In [ ]:
def load_sweep_results() -> pd.DataFrame:
    rows = []
    scenario_meta = SCENARIOS.set_index("name").to_dict(orient="index") if not SCENARIOS.empty else {}
    for rec in scenario_records:
        exp_root = experiment_root_for_record(rec)
        csv_path = exp_root / "runs" / rec["scenario"] / "diag_sweep_results.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df["scenario"] = rec["scenario"]
            for key, value in scenario_meta.get(rec["scenario"], {}).items():
                df[key] = value
            rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def summarize_curvature(results: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if results.empty:
        return pd.DataFrame()
    for scenario, group in results.dropna(subset=["W_J", "dz_m"]).groupby("scenario"):
        g = group.sort_values("dz_m")
        meta_cols = ["group", "geometry_key", "variant", "mesh_profile", "sif_mode", "I_eval_A", "r_core_mm", "area_vs_annular"]
        meta = {c: g[c].iloc[0] for c in meta_cols if c in g.columns}
        row = {"scenario": scenario, **meta, "n_points": len(g), "mean_msh_elements": g.get("msh_elements", pd.Series(dtype=float)).mean()}
        center = g.iloc[(g["dz_m"].abs()).argsort()[:1]]
        if not center.empty:
            row["W0_J"] = float(center["W_J"].iloc[0])
        if len(g) >= 3:
            x = g["dz_m"].to_numpy(dtype=float)
            y = g["W_J"].to_numpy(dtype=float)
            coeff = np.polyfit(x, y, deg=2)
            a, b, c = coeff
            row["force_proxy_N_at0"] = -float(b)
            row["curvature_N_per_m"] = float(2 * a)
            row["W_fit0_J"] = float(c)
            row["fit_rmse_J"] = float(np.sqrt(np.mean((np.polyval(coeff, x) - y) ** 2)))
        notes = sorted(str(n) for n in g.get("note", pd.Series(dtype=str)).dropna().unique() if str(n))
        statuses = sorted(str(s) for s in g.get("status", pd.Series(dtype=str)).dropna().unique() if str(s))
        row["statuses"] = ";".join(statuses)
        row["notes"] = ";".join(notes)
        rows.append(row)
    return pd.DataFrame(rows)


def summarize_amb_delta(metrics: pd.DataFrame) -> pd.DataFrame:
    needed = {"geometry_key", "mesh_profile", "sif_mode", "I_eval_A", "force_proxy_N_at0", "curvature_N_per_m", "W0_J"}
    if metrics.empty or not needed.issubset(metrics.columns):
        return pd.DataFrame()
    rows = []
    keys = ["geometry_key", "mesh_profile", "sif_mode"]
    for key, group in metrics.groupby(keys, dropna=False):
        i0 = group[np.isclose(group["I_eval_A"].astype(float), 0.0)]
        i3 = group[np.isclose(group["I_eval_A"].astype(float), BASELINE_I_A)]
        if i0.empty or i3.empty:
            continue
        a = i0.iloc[0]
        b = i3.iloc[0]
        rows.append({
            "geometry_key": key[0], "mesh_profile": key[1], "sif_mode": key[2],
            "dW0_AMB_J": b["W0_J"] - a["W0_J"],
            "dForce_AMB_N": b["force_proxy_N_at0"] - a["force_proxy_N_at0"],
            "dCurvature_AMB_N_per_m": b["curvature_N_per_m"] - a["curvature_N_per_m"],
            "W0_I0_J": a["W0_J"], "W0_I3_J": b["W0_J"],
        })
    return pd.DataFrame(rows)


results = load_sweep_results()
if results.empty:
    print("No hay diag_sweep_results.csv todavia.")
else:
    display(results[["scenario", "tag", "dz_m", "W_J", "msh_elements", "status", "note"]].head(40))

    metrics = summarize_curvature(results)
    amb_delta = summarize_amb_delta(metrics)
    REPORTS_ROOT.mkdir(parents=True, exist_ok=True)
    results.to_csv(REPORTS_ROOT / "all_sweep_results.csv", index=False)
    metrics.to_csv(REPORTS_ROOT / "central_difference_metrics.csv", index=False)
    amb_delta.to_csv(REPORTS_ROOT / "amb_delta_metrics.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for scenario, group in results.groupby("scenario"):
        g = group.sort_values("dz_m")
        x_mm = g["dz_m"] * 1000.0
        axes[0].plot(x_mm, g["W_J"], marker="o", label=scenario)
        axes[1].plot(x_mm, g["msh_elements"], marker="o", label=scenario)
    axes[0].set_xlabel("dz [mm]")
    axes[0].set_ylabel("ElectroMagnetic Field Energy [J]")
    axes[1].set_xlabel("dz [mm]")
    axes[1].set_ylabel("elementos .msh")
    axes[0].legend(fontsize=8)
    axes[1].legend(fontsize=8)
    plt.tight_layout()

    if not metrics.empty:
        display(metrics.sort_values(["group", "geometry_key", "I_eval_A", "mesh_profile", "sif_mode"]))
        fig, axes = plt.subplots(1, 3, figsize=(17, max(4, 0.35 * len(metrics))))
        m = metrics.sort_values("curvature_N_per_m")
        axes[0].barh(m["scenario"], m["W0_J"])
        axes[0].set_xlabel("W(dz=0) [J]")
        axes[1].barh(m["scenario"], m["force_proxy_N_at0"])
        axes[1].set_xlabel("-dW/dz en 0 [N]")
        axes[2].barh(m["scenario"], m["curvature_N_per_m"])
        axes[2].set_xlabel("d2W/dz2 [N/m]")
        for ax in axes:
            ax.tick_params(axis="y", labelsize=8)
        plt.tight_layout()

    if not amb_delta.empty:
        display(amb_delta)
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.bar(amb_delta["geometry_key"], amb_delta["dForce_AMB_N"])
        ax.axhline(0, color="black", linewidth=1)
        ax.set_ylabel("Delta fuerza por AMB [N] (I=3 A - I=0 A)")
        ax.set_xlabel("geometria")
        plt.tight_layout()

    print("CSVs para subir/analizar:")
    print(REPORTS_ROOT / "all_sweep_results.csv")
    print(REPORTS_ROOT / "central_difference_metrics.csv")
    print(REPORTS_ROOT / "amb_delta_metrics.csv")


## Criterio de decisión inicial

- Si quieres que el AMB se note mas, compara `amb_delta_metrics.csv`: no solo energia total, sino `dForce_AMB_N` y `dCurvature_AMB_N_per_m` entre `I=0` e `I=3 A`.
- Si la pastilla tipo Ruben baja mucho el PMB pero aumenta el delta relativo del AMB, puede ser buena candidata aunque el PMB sea mas debil.
- Si `pill_wallgap0p5` se acerca al anillo sin perder visibilidad AMB, es el compromiso fisico mas defendible: respeta el radio real del rotor menos pared.
- Si la pastilla de area equivalente se parece al anillo, entonces la diferencia anterior venia principalmente de volumen de iman, no de la topologia.
- Para malla, acepta un perfil si `W0`, fuerza y curvatura cambian poco entre `balanced_fixed_layers` y `focus_inner_coil`; si cambian mucho, hay que seguir refinando solo rotor/bobina.
- Para SIF, el candidato fijo es `clean_solver5_quiet_materials_piola_on` si elimina `solver_rc=9` sin mover energia/fuerza de forma relevante. `piola_off` es solo prueba de sensibilidad.
- No compares escenarios hasta que el preflight haya corregido `ExternalBC` por geometria; en las corridas anteriores ya vimos que los IDs cambian entre anillo y pastilla.
